<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/CNN_Examples/CNN_Demo3_GradCAM_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 3: Grad-CAM — Where Does the CNN Look?

**ELC5365 Deep Learning** | Baylor University

CNNs achieve remarkable accuracy, but HOW do they make decisions?
**Grad-CAM** (Gradient-weighted Class Activation Mapping) lets us visualize which
regions of an image are most important for the CNN's prediction.

We will:
1. Implement Grad-CAM from scratch (educational!)
2. Visualize attention heatmaps overlaid on images
3. Compare attention at different layers (shallow vs deep)
4. Show how attention shifts for different target classes
5. Connect Grad-CAM to classification with localization

**Reference:** Selvaraju et al., "Grad-CAM: Visual Explanations from Deep Networks," *ICCV*, 2017.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Section 1: Implementing Grad-CAM from Scratch

**How Grad-CAM works:**
1. Forward pass: capture feature maps at the target layer
2. Backward pass: compute gradients of the target class score w.r.t. feature maps
3. Compute importance weights by **global average pooling** the gradients
4. Compute weighted combination of feature maps
5. Apply **ReLU** (we only want features with positive influence)
6. Upsample to input image size

In [ ]:
class GradCAM:
    """Grad-CAM: Gradient-weighted Class Activation Mapping.
    
    Works with any CNN model and any target convolutional layer.
    """
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.target_layer = target_layer
        self.feature_maps = None
        self.gradients = None
        
        # Register hooks to capture feature maps and gradients
        target_layer.register_forward_hook(self._save_feature_maps)
        target_layer.register_full_backward_hook(self._save_gradients)
    
    def _save_feature_maps(self, module, input, output):
        """Forward hook: save the feature maps."""
        self.feature_maps = output.detach()
    
    def _save_gradients(self, module, grad_input, grad_output):
        """Backward hook: save the gradients."""
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        """Generate Grad-CAM heatmap.
        
        Args:
            input_tensor: preprocessed input image (1, C, H, W)
            target_class: class index to explain (None = use predicted class)
        Returns:
            heatmap: numpy array (H, W) with values in [0, 1]
        """
        # Step 1: Forward pass
        output = self.model(input_tensor)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # Step 2: Backward pass for target class
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Step 3: Global average pooling of gradients -> importance weights
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        
        # Step 4: Weighted combination of feature maps
        cam = (weights * self.feature_maps).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        
        # Step 5: ReLU — only positive influence
        cam = F.relu(cam)
        
        # Step 6: Upsample to input size
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        
        # Normalize to [0, 1]
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam, target_class, output

print("GradCAM class defined. It works with ANY CNN model and ANY target layer.")

---
## Section 2: Loading Model and Test Images

In [ ]:
# Load pretrained ResNet-50
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device)
resnet50.eval()

# ImageNet class labels
IMAGENET_LABELS = models.ResNet50_Weights.IMAGENET1K_V1.meta['categories']

print(f"ResNet-50 loaded with {len(IMAGENET_LABELS)} ImageNet classes.")
print(f"Target layer for Grad-CAM: resnet50.layer4[-1] (last conv block)")

In [ ]:
# Load CIFAR-10 test images as our demo images
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
cifar_classes = cifar10.classes

# Preprocessing for ResNet-50
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_display_image(pil_img):
    """Resize PIL image for display."""
    return pil_img.resize((224, 224), Image.BILINEAR)

# Pick diverse images: airplane, car, bird, cat, deer, dog
sample_indices = [0, 1, 2, 3, 4, 5]
sample_images = []
for idx in sample_indices:
    img, label = cifar10[idx]
    sample_images.append((img, label, cifar_classes[label]))

# Show sample images
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, (img, label, name) in zip(axes, sample_images):
    ax.imshow(get_display_image(img))
    ax.set_title(f'CIFAR: {name}', fontsize=11)
    ax.axis('off')
plt.suptitle('Test Images for Grad-CAM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3: Grad-CAM on Different Images

For each image, we show: **Original | Heatmap | Overlay**

The heatmap reveals which regions the CNN considers most important for its prediction.

In [ ]:
def apply_heatmap(img_np, heatmap, alpha=0.5):
    """Overlay heatmap on image."""
    colormap = plt.cm.jet(heatmap)[:, :, :3]  # Apply jet colormap
    overlay = alpha * colormap + (1 - alpha) * img_np / 255.0
    overlay = np.clip(overlay, 0, 1)
    return overlay

fig, axes = plt.subplots(6, 3, figsize=(10, 20))
fig.suptitle('Grad-CAM: Where Does ResNet-50 Look?', fontsize=18, fontweight='bold')
axes[0, 0].set_title('Original', fontsize=13, fontweight='bold')
axes[0, 1].set_title('Grad-CAM Heatmap', fontsize=13, fontweight='bold')
axes[0, 2].set_title('Overlay', fontsize=13, fontweight='bold')

for row, (img_pil, label, cifar_name) in enumerate(sample_images):
    # Create a fresh model instance for each to avoid hook accumulation
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
    grad_cam = GradCAM(model, model.layer4[-1].conv3)
    
    input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
    input_tensor.requires_grad_(True)
    heatmap, pred_class, output = grad_cam.generate(input_tensor)
    
    confidence = torch.softmax(output, dim=1)[0, pred_class].item()
    pred_name = IMAGENET_LABELS[pred_class]
    
    display_img = np.array(get_display_image(img_pil))
    
    axes[row, 0].imshow(display_img)
    axes[row, 0].set_ylabel(f'CIFAR: {cifar_name}', fontsize=11, rotation=0, labelpad=60, va='center')
    axes[row, 1].imshow(heatmap, cmap='jet')
    axes[row, 2].imshow(apply_heatmap(display_img, heatmap))
    axes[row, 2].set_xlabel(f'Pred: {pred_name} ({confidence:.0%})', fontsize=10)
    
    for ax in axes[row]:
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

print("The heatmap shows which spatial regions contribute most to the prediction.")
print("Bright (red/yellow) regions = high importance. Dark (blue) = low importance.")

---
## Section 4: Grad-CAM at Different Layers

Applying Grad-CAM at different depths reveals a hierarchy:
- **Early layers** highlight fine-grained details (edges, textures)
- **Deep layers** highlight semantically meaningful regions (object parts)

In [ ]:
# Pick one image to analyze across layers
img_pil, label, cifar_name = sample_images[3]  # Try different indices
display_img = np.array(get_display_image(img_pil))

layers_to_examine = [
    ('layer1[-1]', 'layer1'),
    ('layer2[-1]', 'layer2'),
    ('layer3[-1]', 'layer3'),
    ('layer4[-1]', 'layer4'),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f'Grad-CAM at Different Depths (Image: {cifar_name})', fontsize=16, fontweight='bold')

axes[0].imshow(display_img)
axes[0].set_title('Original', fontsize=12)
axes[0].axis('off')

for i, (layer_desc, layer_name) in enumerate(layers_to_examine):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
    target_layer = getattr(model, layer_name)[-1]
    # Use the last conv in the block
    if hasattr(target_layer, 'conv3'):
        hook_layer = target_layer.conv3
    else:
        hook_layer = target_layer.conv2
    
    grad_cam = GradCAM(model, hook_layer)
    input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
    input_tensor.requires_grad_(True)
    heatmap, _, _ = grad_cam.generate(input_tensor)
    
    axes[i+1].imshow(apply_heatmap(display_img, heatmap))
    axes[i+1].set_title(layer_desc, fontsize=12)
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

print("Deeper layers capture more semantically meaningful regions.")
print("layer4 (last conv block) is most commonly used for Grad-CAM.")

---
## Section 5: Grad-CAM for Different Target Classes

We can ask: "If we force the model to explain a DIFFERENT class, where does it look?"

This shows the model has learned to localize different objects/features.

In [ ]:
img_pil, label, cifar_name = sample_images[0]
display_img = np.array(get_display_image(img_pil))

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()

# Get top-3 predictions
input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)
    top3_prob, top3_idx = probs.topk(3)

print("Top 3 predictions:")
for i in range(3):
    print(f"  {i+1}. {IMAGENET_LABELS[top3_idx[0,i].item()]} ({top3_prob[0,i].item():.1%})")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f'Grad-CAM for Different Target Classes (CIFAR: {cifar_name})',
             fontsize=14, fontweight='bold')

axes[0].imshow(display_img)
axes[0].set_title('Original', fontsize=12)
axes[0].axis('off')

for i in range(3):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
    grad_cam = GradCAM(model, model.layer4[-1].conv3)
    input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
    input_tensor.requires_grad_(True)
    
    target_class = top3_idx[0, i].item()
    heatmap, _, _ = grad_cam.generate(input_tensor, target_class=target_class)
    
    axes[i+1].imshow(apply_heatmap(display_img, heatmap))
    axes[i+1].set_title(f'{IMAGENET_LABELS[target_class]}\n({top3_prob[0,i].item():.1%})', fontsize=11)
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

print("Notice how attention shifts when we ask about different classes!")

---
## Section 6: VGG-16 vs ResNet-50 — Do They Look at the Same Things?

In [ ]:
img_pil, label, cifar_name = sample_images[3]
display_img = np.array(get_display_image(img_pil))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle(f'VGG-16 vs ResNet-50 Attention (Image: {cifar_name})', fontsize=14, fontweight='bold')

axes[0].imshow(display_img)
axes[0].set_title('Original', fontsize=12)
axes[0].axis('off')

# VGG-16 Grad-CAM
vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).to(device).eval()
grad_cam_vgg = GradCAM(vgg, vgg.features[28])  # Last conv layer
input_t = preprocess(img_pil).unsqueeze(0).to(device)
input_t.requires_grad_(True)
hm_vgg, cls_vgg, _ = grad_cam_vgg.generate(input_t)
axes[1].imshow(apply_heatmap(display_img, hm_vgg))
axes[1].set_title(f'VGG-16: {IMAGENET_LABELS[cls_vgg]}', fontsize=11)
axes[1].axis('off')

# ResNet-50 Grad-CAM
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
grad_cam_res = GradCAM(resnet, resnet.layer4[-1].conv3)
input_t = preprocess(img_pil).unsqueeze(0).to(device)
input_t.requires_grad_(True)
hm_res, cls_res, _ = grad_cam_res.generate(input_t)
axes[2].imshow(apply_heatmap(display_img, hm_res))
axes[2].set_title(f'ResNet-50: {IMAGENET_LABELS[cls_res]}', fontsize=11)
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("Different architectures may focus on different features for the same image!")

---
## Section 7: From Grad-CAM to Localization

Grad-CAM essentially provides a rough **localization** of the object — without
any bounding box training! This connects to the lecture topic of
**classification with localization**.

By thresholding the heatmap, we can extract a bounding box.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Grad-CAM as Rough Object Localization', fontsize=16, fontweight='bold')

for idx, (img_pil, label, cifar_name) in enumerate(sample_images[:6]):
    ax = axes[idx // 3, idx % 3]
    display_img = np.array(get_display_image(img_pil))
    
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
    grad_cam = GradCAM(model, model.layer4[-1].conv3)
    input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
    input_tensor.requires_grad_(True)
    heatmap, pred_class, _ = grad_cam.generate(input_tensor)
    
    # Threshold at 50% of max to get rough bounding box
    mask = heatmap > 0.5
    if mask.any():
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
    else:
        rmin, rmax, cmin, cmax = 0, 223, 0, 223
    
    ax.imshow(display_img)
    rect = patches.Rectangle((cmin, rmin), cmax-cmin, rmax-rmin,
                              linewidth=3, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    pred_name = IMAGENET_LABELS[pred_class]
    ax.set_title(f'{pred_name}', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Green boxes: rough localization from Grad-CAM heatmap (threshold = 50%)")
print("\nThis is 'free' localization without training any bounding box regressor!")
print("For precise localization, we need dedicated detection models (R-CNN, YOLO, etc.)")

---
## Summary

| Technique | What it shows |
|-----------|---------------|
| **Grad-CAM on different images** | Which spatial regions drive each prediction |
| **Grad-CAM at different layers** | Shallow = edges, deep = semantic regions |
| **Grad-CAM for different classes** | The model learns to localize different objects |
| **Grad-CAM as localization** | "Free" bounding boxes without detection training |

**Key takeaway:** Grad-CAM reveals that CNNs learn meaningful spatial features,
not just texture statistics. This builds trust in model predictions and connects
classification to localization.